# Fair Feeder v14 — Full Training Pipeline

This notebook runs **entirely online** (Colab or Kaggle). No local setup needed.

**What it does:**
1. Installs dependencies
2. Downloads v14 dataset from Roboflow
3. Fixes polygon labels → bounding boxes
4. Visual verification (you check the boxes look right)
5. Trains YOLOv11 with W&B tracking + checkpoint resume
6. Reviews results and downloads trained model

**Before you start:** Make sure you have a GPU runtime enabled:
- **Colab**: Runtime → Change runtime type → T4 GPU
- **Kaggle**: Settings → Accelerator → GPU T4 x2

**If your session disconnects:** Skip to Step 5b to resume from the last checkpoint.

## Step 0 — Install Dependencies

In [ ]:
!pip install -q ultralytics roboflow opencv-python-headless wandb

## Step 1 — Set Your API Keys

Replace the placeholder values below with your actual keys.

In [ ]:
ROBOFLOW_API_KEY = "<REDACTED_SECRET>"  # <-- paste your Roboflow key here
WANDB_API_KEY = "<REDACTED_SECRET>"  # <-- paste your W&B key here (from wandb.ai/authorize)!pip install roboflow

## Step 1b — Setup W&B + Persistent Storage

This logs all metrics to W&B and saves checkpoints to Google Drive (Colab) or `/kaggle/working/` (Kaggle) so you can resume if disconnected.

In [ ]:
import os
from pathlib import Path

# --- W&B login ---
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
import wandb
wandb.login()
print("W&B: logged in")

# --- Persistent storage for checkpoints ---
# Check Kaggle FIRST — Kaggle has google.colab installed but drive.mount() fails there
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ON_COLAB = not ON_KAGGLE and "COLAB_RELEASE_TAG" in os.environ

if ON_KAGGLE:
    SAVE_DIR = Path("/kaggle/working/fair-feeder-v14")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Checkpoints will save to: {SAVE_DIR}")
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/fair-feeder-v14")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Checkpoints will sync to Google Drive: {SAVE_DIR}")
else:
    SAVE_DIR = Path("runs/fair-feeder/v14/weights")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Checkpoints will save to: {SAVE_DIR}")

print("Setup complete.")

## Step 2 — Download v14 Dataset from Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("test-7vyqo").project("ir-kibble")
version = project.version(14)
dataset = version.download("yolov8")

DATASET_PATH = dataset.location
print(f"\nDataset downloaded to: {DATASET_PATH}")

## Step 3 — Fix Labels (Polygon → Bounding Box)

Some labels from Roboflow may be polygon outlines instead of rectangles.
This converts them and verifies zero polygons remain.

In [ ]:
import shutil
from pathlib import Path


def is_polygon_line(line):
    return len(line.strip().split()) > 5


def polygon_to_bbox(line):
    parts = line.strip().split()
    class_id = parts[0]
    coords = [float(v) for v in parts[1:]]
    xs = coords[0::2]
    ys = coords[1::2]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = x_max - x_min
    height = y_max - y_min
    return f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"


dataset_path = Path(DATASET_PATH)
total_converted = 0

for split in ["train", "valid", "test"]:
    label_dir = dataset_path / split / "labels"
    if not label_dir.is_dir():
        print(f"[{split}] No labels directory — skipping")
        continue

    # Backup originals
    backup_dir = dataset_path / split / "labels_backup_polygon"
    if not backup_dir.exists():
        shutil.copytree(label_dir, backup_dir)
        print(f"[{split}] Backup created: {backup_dir}")

    # Convert
    split_converted = 0
    for f in sorted(label_dir.glob("*.txt")):
        lines = f.read_text().splitlines()
        new_lines = []
        for line in lines:
            if not line.strip():
                continue
            if is_polygon_line(line):
                new_lines.append(polygon_to_bbox(line))
                split_converted += 1
            else:
                new_lines.append(line.strip())
        f.write_text("\n".join(new_lines) + "\n" if new_lines else "")

    # Verify
    remaining = sum(
        1 for f in label_dir.glob("*.txt")
        for line in f.read_text().splitlines()
        if line.strip() and is_polygon_line(line)
    )
    status = "PASS" if remaining == 0 else f"FAIL ({remaining} polygons left)"
    print(f"[{split}] Converted {split_converted} polygon lines — Verification: {status}")
    total_converted += split_converted

print(f"\nTotal polygon lines converted: {total_converted}")

## Step 4 — Visual Label Check

Look at 12 random images with bounding boxes drawn on them.
If the boxes are sitting on the right objects, you're good to train.

**Colour legend:**
- Blue = Dan
- Orange = Sanbo
- Green = Dan_hand (paw)
- Yellow = Bowl
- Magenta = Kibble

In [ ]:
import math
import random

import cv2
import numpy as np
from IPython.display import display
from PIL import Image

CLASS_NAMES = ["Dan", "Sanbo", "Dan_hand", "Bowl", "Kibble"]
CLASS_COLORS = {
    0: (255, 100, 0),    # Dan — Blue (BGR)
    1: (0, 165, 255),    # Sanbo — Orange
    2: (0, 255, 0),      # Dan_hand — Green
    3: (0, 255, 255),    # Bowl — Yellow
    4: (255, 0, 255),    # Kibble — Magenta
}

THUMB_W, THUMB_H = 320, 180  # 16:9 to match 1280x720 source images
GRID_COLS = 4
NUM_SAMPLES = 12

image_dir = dataset_path / "train" / "images"
label_dir = dataset_path / "train" / "labels"

image_files = sorted(
    p for p in image_dir.iterdir()
    if p.suffix.lower() in (".jpg", ".jpeg", ".png")
)
sampled = random.sample(image_files, min(NUM_SAMPLES, len(image_files)))

thumbnails = []
for img_path in sampled:
    image = cv2.imread(str(img_path))
    if image is None:
        continue
    label_path = label_dir / (img_path.stem + ".txt")
    if label_path.exists():
        img_h, img_w = image.shape[:2]
        for line in label_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls = int(parts[0])
            x_ctr, y_ctr, w, h = map(float, parts[1:])
            x1 = int((x_ctr - w / 2) * img_w)
            y1 = int((y_ctr - h / 2) * img_h)
            x2 = int((x_ctr + w / 2) * img_w)
            y2 = int((y_ctr + h / 2) * img_h)
            color = CLASS_COLORS.get(cls, (255, 255, 255))
            cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
            name = CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else str(cls)
            cv2.putText(image, name, (x1 + 2, y1 - 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    thumbnails.append(cv2.resize(image, (THUMB_W, THUMB_H)))

# Pad to fill grid
rows = math.ceil(len(thumbnails) / GRID_COLS)
while len(thumbnails) < rows * GRID_COLS:
    thumbnails.append(np.zeros((THUMB_H, THUMB_W, 3), dtype=np.uint8))

grid_rows = []
for r in range(rows):
    grid_rows.append(np.hstack(thumbnails[r * GRID_COLS:(r + 1) * GRID_COLS]))
grid = np.vstack(grid_rows)

# Convert BGR → RGB for display in notebook
grid_rgb = cv2.cvtColor(grid, cv2.COLOR_BGR2RGB)
display(Image.fromarray(grid_rgb))

print("\nIf boxes are on the right objects, proceed to training!")
print("If something looks wrong, check your Roboflow annotations.")

## Step 5a — Train YOLOv11 (Fresh Start)

All V12 bug fixes baked in. W&B tracks every metric in real-time.
After training, checkpoints are copied to persistent storage.

**If disconnected:** Skip this cell and go to **Step 5b** instead.

In [ ]:
import torch
from ultralytics import YOLO

# Check GPU
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = 0
else:
    print("WARNING: No GPU detected! Training will be very slow.")
    print("Enable GPU: Colab → Runtime → Change runtime type → T4 GPU")
    device = "cpu"

# Use the data.yaml that Roboflow downloaded with the dataset
data_yaml = str(dataset_path / "data.yaml")
print(f"Dataset config: {data_yaml}")

# Enable W&B logging for YOLO
os.environ["WANDB_PROJECT"] = "Fair-Feeder-YOLO11"
os.environ["WANDB_ENTITY"] = "iknoest-hknlpost"

In [ ]:
import shutil

model = YOLO("yolo11s.pt")

results = model.train(
    data=data_yaml,
    imgsz=[1280, 720],         # Exact Roboflow export size (16:9) — no resize distortion
    rect=True,                 # Batches by aspect ratio — no letterbox padding

    # Geometric augmentations
    fliplr=0.5,             # Horizontal flip
    flipud=0.0,             # OFF — fixed overhead camera
    degrees=0.0,            # Rotation OFF — V12 root cause of coordinate drift

    # Colour — only brightness matters for IR grayscale
    hsv_h=0.0,              # OFF — no hue in grayscale
    hsv_s=0.0,              # OFF — no saturation in grayscale
    hsv_v=0.25,             # ON  — random brightness variation

    # Advanced augmentations
    mosaic=1.0,             # Combines 4 images — good for small objects
    copy_paste=0.0,         # Kibble class balancing #v14=0.3
    mixup=0.0,              # OFF — destroys small kibble detail

    # Training config
    batch=8,
    epochs=100,
    patience=20,            # Early stopping if no improvement for 20 epochs
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,

    # Output
    project="runs/fair-feeder",
    name="v14",
    device=device,
    verbose=True,
    plots=True,
)

# Copy checkpoints to persistent storage (survives disconnect)
weights_dir = Path("runs/fair-feeder/v14/weights")
if weights_dir.exists():
    for pt_file in weights_dir.glob("*.pt"):
        shutil.copy2(pt_file, SAVE_DIR / pt_file.name)
    print(f"\nCheckpoints saved to: {SAVE_DIR}")
    for f in sorted(SAVE_DIR.glob("*.pt")):
        print(f"  {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

print("\nTraining complete! Check W&B for full metrics.")

## Step 5b — Resume Training After Disconnect (auto-skips on first run)

In [ ]:
import shutil
from ultralytics import YOLO

# Auto-skip if training already completed (first run via "Run All")
best_pt = SAVE_DIR / "best.pt"
last_pt = SAVE_DIR / "last.pt"

if best_pt.exists() and not last_pt.exists():
    print("Training already completed — skipping resume.")
elif not last_pt.exists():
    print("No checkpoint found — skipping (run Step 5a first).")
else:
    # Restore checkpoint to local working directory
    local_weights = Path("runs/fair-feeder/v14/weights")
    local_weights.mkdir(parents=True, exist_ok=True)
    shutil.copy2(last_pt, local_weights / "last.pt")
    print(f"Restored checkpoint from: {last_pt}")

    model = YOLO(str(local_weights / "last.pt"))
    results = model.train(resume=True)

    # Save final checkpoints back to persistent storage
    for pt_file in local_weights.glob("*.pt"):
        shutil.copy2(pt_file, SAVE_DIR / pt_file.name)
    print(f"\nFinal checkpoints saved to: {SAVE_DIR}")
    print("Training complete! Check W&B for full metrics.")

## Step 6 — Review Results

Check the training metrics and confusion matrix.

In [ ]:
from IPython.display import Image as IPImage, display

results_dir = Path("runs/fair-feeder/v14")

# Show confusion matrix
confusion_matrix = results_dir / "confusion_matrix.png"
if confusion_matrix.exists():
    print("Confusion Matrix:")
    display(IPImage(filename=str(confusion_matrix)))

# Show training curves
results_plot = results_dir / "results.png"
if results_plot.exists():
    print("\nTraining Curves:")
    display(IPImage(filename=str(results_plot)))

# Show best model path
best_model = results_dir / "weights" / "best.pt"
if best_model.exists():
    print(f"\nBest model saved at: {best_model}")
    print(f"Model size: {best_model.stat().st_size / 1024 / 1024:.1f} MB")
    print("\nDownload this file to use in your Fair Feeder application.")

## Step 7 — Download Your Trained Model

Run this cell to download the `best.pt` model file to your computer.

In [ ]:
import os

best_model = Path("runs/fair-feeder/v14/weights/best.pt")

if not best_model.exists():
    print("Model not found — make sure training completed successfully.")
else:
    # Detect platform and trigger download
    if "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import files
        files.download(str(best_model))
        print("Download started (check your browser downloads).")
    elif "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        # On Kaggle, save to /kaggle/working/ for the Output tab
        import shutil
        output_path = Path("/kaggle/working/best.pt")
        shutil.copy2(best_model, output_path)
        print(f"Model copied to {output_path}")
        print("Go to the Output tab to download it.")
    else:
        print(f"Model is at: {best_model}")
        print(f"Size: {best_model.stat().st_size / 1024 / 1024:.1f} MB")